# **Data Loader test**
---

### Initialize libraries and modules:

In [ ]:
import sys
import itertools
from pathlib import Path
import numpy as np
import pandas as pd

class ImportMyModules:

    def __init__(self) -> None:
        self.cwd = Path.cwd().resolve()

    def _get_repo_root(self, folder: str) -> Path:
        return next(
            parent for parent in [self.cwd, *self.cwd.parents]
            if (parent / "lib" / "peregrin" / folder).exists()
        )

    def insert_path(self, folder: str) -> None:
        repo_root = self._get_repo_root("src")

        package_root = repo_root / "lib" / "peregrin"
        print(f"Adding {package_root} to sys.path")
        sys.path.insert(0, str(package_root))

importer = ImportMyModules()
importer.insert_path("src")
importer.insert_path("data")

Adding C:\Users\modri\Desktop\Repositories\peregrin\lib\peregrin to sys.path
Adding C:\Users\modri\Desktop\Repositories\peregrin\lib\peregrin to sys.path


#### A helper for dummy dataset generation:

In [19]:
def craft_dummy_dataframe(
    n_tracks: int = 6,
    n_time_points: int = 100,
    seed: int | None = 20,
    categories: dict[str, list[str]] | None = None,
    save: bool | str = False,          # False | 'single' | 'split' | 'tree'
    out_dir: str | Path = ".",
    filename: str = "dummy_cell_tracks",
    n_files: int = 1,                  # files per leaf ('tree'/'split' modes)
) -> tuple[pd.DataFrame, dict | str | None]:
    """
    Generate synthetic persistent-random-walk cell trajectories, organised
    into an arbitrary category hierarchy.

    Parameters
    ----------
    categories : dict, optional
        Ordered mapping of category level -> labels, e.g.::

            {
                'set':      ['set_A', 'set_B'],
                'subset':   ['subset_1', 'subset_2'],
                'group':    ['ctrl', 'treated'],
                'subgroup': ['rep1', 'rep2'],
            }

        Any subset of levels works (1-4). Defaults to 2 sets x 2 subsets.
        `n_tracks` trajectories are generated per leaf combination.

    save :
        - False    -> no files written
        - 'single' -> one CSV with all category columns
        - 'split'  -> one CSV per category combination (flat directory),
                      named e.g. dummy_cell_tracks__set_A__subset_1.csv
        - 'tree'   -> directory tree out_dir/set_A/subset_1/.../<filename>.csv

    Returns
    -------
    (df, paths)
        `df` is the combined DataFrame. `paths` is:
        - 'single' -> the file path (str)
        - 'split'/'tree' -> a nested dict of file paths mirroring the
          category hierarchy, directly consumable by `load_data(paths)`
        - False -> None
    """
    if categories is None:
        categories = {
            'set': ['set_A', 'set_B'],
            'subset': ['subset_1', 'subset_2'],
        }

    allowed = ['set', 'subset', 'group', 'subgroup']
    levels = [lvl for lvl in allowed if lvl in categories]
    if not levels:
        raise ValueError(f"categories must use keys from {allowed}")

    rng = np.random.default_rng(seed)
    records = []
    global_track_id = 0

    def _simulate_track(track_id: int) -> list[tuple]:
        start = rng.integers(0, n_time_points // 4)
        length = rng.integers(n_time_points // 2, n_time_points + 1)
        end = min(start + length, n_time_points)

        x, y = rng.uniform(0, 500), rng.uniform(0, 500)
        speed = rng.uniform(2.0, 8.0)
        persistence = rng.uniform(0.6, 0.98)
        angle = rng.uniform(0, 2 * np.pi)

        rows = []
        for t in range(start, end):
            rows.append((track_id, t, x, y))
            angle += (1 - persistence) * rng.uniform(-np.pi, np.pi)
            step = rng.normal(speed, speed * 0.2)
            x += step * np.cos(angle) + rng.normal(0, 0.5)
            y += step * np.sin(angle) + rng.normal(0, 0.5)
        return rows

    # Cartesian product over all category levels -> one leaf per combination
    combos = list(itertools.product(*(categories[lvl] for lvl in levels)))
    for combo in combos:
        for _ in range(n_tracks):
            global_track_id += 1
            for track_id, t, x, y in _simulate_track(global_track_id):
                records.append(combo + (track_id, t, x, y))

    df = pd.DataFrame(
        records,
        columns=levels + ['track_id', 'time_point', 'x_coordinate', 'y_coordinate'],
    )
    df = df.sort_values(levels + ['track_id', 'time_point']).reset_index(drop=True)
    df['track_uid'] = df.groupby(levels + ['track_id']).ngroup()

    def clear_directory(path: Path) -> None:
        """Delete all files and subdirectories in the given directory."""
        if path.exists() and path.is_dir():
            for item in path.iterdir():
                if item.is_file():
                    item.unlink()
                elif item.is_dir():
                    clear_directory(item)
                    item.rmdir()

    def _split_tracks(sub: pd.DataFrame) -> list[pd.DataFrame]:
        """Split a leaf's tracks into up to n_files roughly equal chunks."""
        ids = sub['track_id'].unique()
        chunks = np.array_split(ids, min(n_files, len(ids)))
        return [sub[sub['track_id'].isin(chunk)] for chunk in chunks if len(chunk)]

    # ------------------------------------------------------------------ save
    paths = None
    out_dir = Path(out_dir)

    out_dir.mkdir(parents=True, exist_ok=True)

    if save in (True, 'single'):
        single_dir = out_dir / 'single'
        clear_directory(single_dir)
        single_dir.mkdir(parents=True, exist_ok=True)
        path = single_dir / f"{filename}.csv"
        df.to_csv(path, index=False)
        paths = str(path)

    elif save == 'split':
        paths = {}
        for combo, sub in df.groupby(levels, sort=False):
            combo = combo if isinstance(combo, tuple) else (combo,)
            stem = filename + "".join(f"__{c}" for c in combo)

            split_dir = out_dir / 'split'
            clear_directory(split_dir)
            split_dir.mkdir(parents=True, exist_ok=True)
            files = []
            for j, part in enumerate(_split_tracks(sub)):
                path = split_dir / f"{stem}_{j}.csv"
                part.drop(columns=levels).to_csv(path, index=False)
                files.append(str(path))
            node = paths
            for c in combo[:-1]:
                node = node.setdefault(c, {})
            node[combo[-1]] = files if n_files > 1 else files[0]

    elif save == 'tree':
        paths = {}
        tree_dir = out_dir / 'tree'
        clear_directory(tree_dir)

        for combo, sub in df.groupby(levels, sort=False):
            combo = combo if isinstance(combo, tuple) else (combo,)
            leaf_dir = tree_dir.joinpath(*combo)
            leaf_dir.mkdir(parents=True, exist_ok=True)
            files = []
            for j, part in enumerate(_split_tracks(sub)):
                path = leaf_dir / f"{filename}_{j}.csv"
                part.drop(columns=levels).to_csv(path, index=False)
                files.append(str(path))
            node = paths
            for c in combo[:-1]:
                node = node.setdefault(c, {})
            node[combo[-1]] = files if n_files > 1 else files[0]

    elif save is not False:
        raise ValueError(f"Invalid save mode: {save!r}. Use False, 'single', 'split', or 'tree'.")

    return df, paths

### Define paths of files or directories from which data will be loaded:

Merged csv with included (sub)categories (no metadata)

In [26]:
df, path_simulated_csv = craft_dummy_dataframe(
    categories={
        'set': ['set_A', 'set_B'],
        'subset': ['subset_1', 'subset_2'],
    },
    save='single',
    out_dir='dummy_data',
)

Tree like directory with multiple (sub)categories and files (no metadata)

In [ ]:
# Full 4-level hierarchy written as a directory tree
df, _ = craft_dummy_dataframe(
    n_tracks=3,
    categories={
        'set': ['set_A', 'set_B'],
        'subset': ['subset_1', 'subset_2'],
        'group': ['ctrl', 'treated'],
        'subgroup': ['rep1', 'rep2'],
    },
    save='tree',
    out_dir='dummy_data',
    n_files=3,
    seed=42
)

path_simulated_tree = r"dummy_data/tree"

In [22]:
path_trackmate_xml = r"C:\Users\modri\Desktop\img_sq.xml"

In [23]:
path_trackmate_csv = r"C:\Users\modri\Desktop\position_0003440_allspots.csv"

In [24]:
import src.loader.load as load_module
from importlib import reload

reload(load_module)
load_data = load_module.load_data

In [8]:
data_xml = load_data(path_xml)

In [9]:
data_csv = load_data(path_csv)

In [10]:
data_csv_converted = load_data(
    path_csv,
    convert_spatial_to='nm', convert_time_to='min'
)

In [12]:
path_csv_naked = r".\dummy_data\single\dummy_cell_tracks.csv"

data_csv_naked = load_data(
    path_csv_naked, 
    {
        'id': 'track_id',
        't': 'time_point',
        'x': 'x_coordinate',
        'y': 'y_coordinate',
    },
)

C:\Users\modri\Desktop\Repositories\peregrin\lib\peregrin\src\loader\load.py:120: InputWarning: No time units found in input files.
 Please specify the time units using <load_data result>.metadata.write(time_unit="<unit>")
  self._check()
C:\Users\modri\Desktop\Repositories\peregrin\lib\peregrin\src\loader\load.py:120: InputWarning: No spatial units found in input files.
 Please specify the spatial units using <load_data result>.metadata.write(spatial_unit="<unit>")
  self._check()


In [13]:
data_csv_naked.metadata.write(spatialunits='microns', timeunits='seconds')

In [14]:
data_csv_custom = load_data(
    path_csv_naked, 
    {
        'id': 'track_id',
        't': 'time_point',
        'x': 'x_coordinate',
        'y': 'y_coordinate',
    },
    spatial_unit='meter', time_unit='sec'
)

In [15]:
data = [data_xml, data_csv, data_csv_converted, data_csv_naked, data_csv_custom]

In [16]:
for d in data:
    display(d)
    print(d.metadata.get())
    print(d.metadata.get_each())

    assert d.metadata.get('spatialunits') in ['nm', 'μm', 'mm', 'cm', 'm']
    assert d.metadata.get('timeunits') in ['ns', 'μs', 'ms', 's', 'min', 'h', 'd']

track_id,time_point,x_coordinate,y_coordinate,set
f64,f64,f64,f64,str
149.0,0.0,107.037336,0.0,"""img_sq.xml"""
153.0,0.0,618.601983,0.0,"""img_sq.xml"""
154.0,0.0,669.536991,0.0,"""img_sq.xml"""
155.0,0.0,720.471999,0.0,"""img_sq.xml"""
156.0,0.0,971.456097,0.0,"""img_sq.xml"""
…,…,…,…,…
141.0,8010.0,668.458968,875.951429,"""img_sq.xml"""
1529.0,8010.0,830.026108,879.125552,"""img_sq.xml"""
1577.0,8010.0,487.942614,885.08804,"""img_sq.xml"""


{'spatialunits': 'μm', 'timeunits': 's', 'timeinterval': 90.0, 'nframes': 90, 'columns': ['TRACK_ID', 'ID', 'name', 'STD_INTENSITY_CH1', 'QUALITY', 'POSITION_T', 'MIN_INTENSITY_CH1', 'TOTAL_INTENSITY_CH1', 'CONTRAST_CH1', 'SNR_CH1', 'FRAME', 'MEDIAN_INTENSITY_CH1', 'VISIBILITY', 'RADIUS', 'POSITION_X', 'POSITION_Y', 'MEAN_INTENSITY_CH1', 'POSITION_Z', 'MAX_INTENSITY_CH1']}
{'img_sq.xml': {'TRACK_ID': '', 'POSITION_T': 's', 'POSITION_X': 'μm', 'POSITION_Y': 'μm', 'spatialunits': 'μm', 'timeunits': 's', 'timeinterval': 90.0, 'nframes': 90, 'columns': ['TRACK_ID', 'ID', 'name', 'STD_INTENSITY_CH1', 'QUALITY', 'POSITION_T', 'MIN_INTENSITY_CH1', 'TOTAL_INTENSITY_CH1', 'CONTRAST_CH1', 'SNR_CH1', 'FRAME', 'MEDIAN_INTENSITY_CH1', 'VISIBILITY', 'RADIUS', 'POSITION_X', 'POSITION_Y', 'MEAN_INTENSITY_CH1', 'POSITION_Z', 'MAX_INTENSITY_CH1']}}


track_id,time_point,x_coordinate,y_coordinate,set
f64,f64,f64,f64,str
149.0,0.0,107.037336,0.0,"""position_0003440_allspots.csv"""
153.0,0.0,618.601983,0.0,"""position_0003440_allspots.csv"""
154.0,0.0,669.536991,0.0,"""position_0003440_allspots.csv"""
155.0,0.0,720.471999,0.0,"""position_0003440_allspots.csv"""
156.0,0.0,971.456097,0.0,"""position_0003440_allspots.csv"""
…,…,…,…,…
141.0,8010.0,668.458968,875.951429,"""position_0003440_allspots.csv"""
1529.0,8010.0,830.026108,879.125552,"""position_0003440_allspots.csv"""
1577.0,8010.0,487.942614,885.08804,"""position_0003440_allspots.csv"""


{'spatialunits': 'μm', 'timeunits': 's', 'timeinterval': 90.0, 'nframes': 90, 'columns': ['LABEL', 'ID', 'TRACK_ID', 'QUALITY', 'POSITION_X', 'POSITION_Y', 'POSITION_Z', 'POSITION_T', 'FRAME', 'RADIUS', 'VISIBILITY', 'MANUAL_SPOT_COLOR', 'MEAN_INTENSITY_CH1', 'MEDIAN_INTENSITY_CH1', 'MIN_INTENSITY_CH1', 'MAX_INTENSITY_CH1', 'TOTAL_INTENSITY_CH1', 'STD_INTENSITY_CH1', 'EXTRACK_P_STUCK', 'EXTRACK_P_DIFFUSIVE', 'CONTRAST_CH1', 'SNR_CH1']}
{'position_0003440_allspots.csv': {'TRACK_ID': '', 'POSITION_T': 's', 'POSITION_X': 'μm', 'POSITION_Y': 'μm', 'spatialunits': 'μm', 'timeunits': 's', 'timeinterval': 90.0, 'nframes': 90, 'columns': ['LABEL', 'ID', 'TRACK_ID', 'QUALITY', 'POSITION_X', 'POSITION_Y', 'POSITION_Z', 'POSITION_T', 'FRAME', 'RADIUS', 'VISIBILITY', 'MANUAL_SPOT_COLOR', 'MEAN_INTENSITY_CH1', 'MEDIAN_INTENSITY_CH1', 'MIN_INTENSITY_CH1', 'MAX_INTENSITY_CH1', 'TOTAL_INTENSITY_CH1', 'STD_INTENSITY_CH1', 'EXTRACK_P_STUCK', 'EXTRACK_P_DIFFUSIVE', 'CONTRAST_CH1', 'SNR_CH1']}}


track_id,time_point,x_coordinate,y_coordinate,set
f64,f64,f64,f64,str
149.0,0.0,107037.335957,0.0,"""position_0003440_allspots.csv"""
153.0,0.0,618601.982978,0.0,"""position_0003440_allspots.csv"""
154.0,0.0,669536.991123,0.0,"""position_0003440_allspots.csv"""
155.0,0.0,720471.999268,0.0,"""position_0003440_allspots.csv"""
156.0,0.0,971456.097374,0.0,"""position_0003440_allspots.csv"""
…,…,…,…,…
141.0,133.5,668458.967563,875951.42899,"""position_0003440_allspots.csv"""
1529.0,133.5,830026.107732,879125.551618,"""position_0003440_allspots.csv"""
1577.0,133.5,487942.614258,885088.040084,"""position_0003440_allspots.csv"""


{'spatialunits': 'μm', 'timeunits': 's', 'timeinterval': 90.0, 'nframes': 90, 'columns': ['LABEL', 'ID', 'TRACK_ID', 'QUALITY', 'POSITION_X', 'POSITION_Y', 'POSITION_Z', 'POSITION_T', 'FRAME', 'RADIUS', 'VISIBILITY', 'MANUAL_SPOT_COLOR', 'MEAN_INTENSITY_CH1', 'MEDIAN_INTENSITY_CH1', 'MIN_INTENSITY_CH1', 'MAX_INTENSITY_CH1', 'TOTAL_INTENSITY_CH1', 'STD_INTENSITY_CH1', 'EXTRACK_P_STUCK', 'EXTRACK_P_DIFFUSIVE', 'CONTRAST_CH1', 'SNR_CH1']}
{'position_0003440_allspots.csv': {'TRACK_ID': '', 'POSITION_T': 's', 'POSITION_X': 'μm', 'POSITION_Y': 'μm', 'spatialunits': 'μm', 'timeunits': 's', 'timeinterval': 90.0, 'nframes': 90, 'columns': ['LABEL', 'ID', 'TRACK_ID', 'QUALITY', 'POSITION_X', 'POSITION_Y', 'POSITION_Z', 'POSITION_T', 'FRAME', 'RADIUS', 'VISIBILITY', 'MANUAL_SPOT_COLOR', 'MEAN_INTENSITY_CH1', 'MEDIAN_INTENSITY_CH1', 'MIN_INTENSITY_CH1', 'MAX_INTENSITY_CH1', 'TOTAL_INTENSITY_CH1', 'STD_INTENSITY_CH1', 'EXTRACK_P_STUCK', 'EXTRACK_P_DIFFUSIVE', 'CONTRAST_CH1', 'SNR_CH1']}, 'spatialun

track_id,time_point,x_coordinate,y_coordinate,set
f64,f64,f64,f64,str
1.0,25.0,242.8987,57.809882,"""dummy_cell_tracks.csv"""
1.0,26.0,245.228029,54.597547,"""dummy_cell_tracks.csv"""
1.0,27.0,249.827267,49.532794,"""dummy_cell_tracks.csv"""
1.0,28.0,252.715365,45.316116,"""dummy_cell_tracks.csv"""
1.0,29.0,253.585434,39.076147,"""dummy_cell_tracks.csv"""
…,…,…,…,…
24.0,95.0,165.663738,651.145494,"""dummy_cell_tracks.csv"""
24.0,96.0,160.947074,653.762386,"""dummy_cell_tracks.csv"""
24.0,97.0,151.214694,656.336399,"""dummy_cell_tracks.csv"""


{'spatialunits': 'μm', 'timeunits': 's', 'timeinterval': 1.0, 'nframes': 99, 'columns': ['set', 'subset', 'track_id', 'time_point', 'x_coordinate', 'y_coordinate', 'track_uid']}
{'dummy_cell_tracks.csv': {'track_id': '', 'time_point': '', 'x_coordinate': '', 'y_coordinate': '', 'spatialunits': '', 'timeunits': '', 'timeinterval': 1.0, 'nframes': 99, 'columns': ['set', 'subset', 'track_id', 'time_point', 'x_coordinate', 'y_coordinate', 'track_uid']}}


track_id,time_point,x_coordinate,y_coordinate,set
f64,f64,f64,f64,str
1.0,25.0,242.8987,57.809882,"""dummy_cell_tracks.csv"""
1.0,26.0,245.228029,54.597547,"""dummy_cell_tracks.csv"""
1.0,27.0,249.827267,49.532794,"""dummy_cell_tracks.csv"""
1.0,28.0,252.715365,45.316116,"""dummy_cell_tracks.csv"""
1.0,29.0,253.585434,39.076147,"""dummy_cell_tracks.csv"""
…,…,…,…,…
24.0,95.0,165.663738,651.145494,"""dummy_cell_tracks.csv"""
24.0,96.0,160.947074,653.762386,"""dummy_cell_tracks.csv"""
24.0,97.0,151.214694,656.336399,"""dummy_cell_tracks.csv"""


{'spatialunits': 'm', 'timeunits': 's', 'timeinterval': 1.0, 'nframes': 99, 'columns': ['set', 'subset', 'track_id', 'time_point', 'x_coordinate', 'y_coordinate', 'track_uid']}
{'dummy_cell_tracks.csv': {'track_id': '', 'time_point': '', 'x_coordinate': '', 'y_coordinate': '', 'spatialunits': 'm', 'timeunits': 's', 'timeinterval': 1.0, 'nframes': 99, 'columns': ['set', 'subset', 'track_id', 'time_point', 'x_coordinate', 'y_coordinate', 'track_uid']}}
